# SFINCS — NJ Sandy: results viewer

Opens a finished run **read-only** and plots it. The adopted premier is `sealed_faber_waves` —
the 2026-07-14 rebuild that **plugged the Navesink leak and carved open Shark River Inlet**
(two DEM/mask defects, not physics; full write-up in `reports/shrewsbury_investigation.md`).

The **Results** section shows each figure **before vs after** that fix — `before` = the old
premier on the broken domain, `after` = `sealed_faber_waves`.

## Setup

In [ ]:
# Viz stack (this import also primes PROJ before hydromt loads).
import sys
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
from hydromt_sfincs import SfincsModel
from nj_sfincs import plots, validate

# The run to view (methodology + wave field come from this one). Adopted premier:
EXP = "sealed_faber_waves"
exp_dir = ROOT / "experiments" / EXP
if not exp_dir.exists():
    exp_dir = ROOT / "model"  # fall back to the reference build
    print(f"experiments/{EXP} not found — showing the reference model/ build")
print("viewing:", exp_dir)

In [ ]:
# Open read-only: `sf` (build + forcing inputs) and `mod` (solver output); downscale once.
sf = SfincsModel(str(exp_dir), data_libs=[str(ROOT / "data" / "data_catalog.yml")], mode="r")
sf.read()
mod, da_hmax, da_dep = validate.load_floodmap(exp_dir)
print("output vars:", list(mod.output.data.keys()))

## Methodology — the build

The sealed run's own grid, topobathy, and mask.

### Quadtree grid

In [ ]:
plots.plot_grid(sf);

### Topobathy (interactive) — pan/zoom the dunes, inlets, dredged channels

In [ ]:
plots.plot_topobathy(sf)

### Mask — active interior, water-level boundary, outflow

In [ ]:
plots.plot_mask(sf);

## Methodology — the forcing

The compound drivers, read back from the written model.

### Surge boundary (NOAA CO-OPS)

In [ ]:
plots.plot_surge(sf);

### Wind + pressure (ERA5)

In [ ]:
plots.plot_wind_pressure(sf);

### Rainfall (NOAA AORC)

In [ ]:
plots.plot_rain(sf);

### River discharge (USGS)

In [ ]:
plots.plot_discharge(sf);

## Results — before vs after the leak + Shark fix

Each figure below puts the old premier (**before**) beside the adopted `sealed_faber_waves`
(**after**). The methodology figures above are the sealed run's own inputs.

In [ ]:
# before / after for the Results panels.
BEFORE = "snapwave_tuned_25m"   # old premier: estuary leaking, Shark inlet dammed shut
AFTER  = "sealed_faber_waves"   # adopted premier: leak sealed, Shark carved open
BA = {"before — broken domain": BEFORE, "after — sealed + carved": AFTER}

### Wave field — SnapWave Hm0 at peak (the premier)

Single panel: the lee behind Sandy Hook in the adopted run.

In [ ]:
res = plots.plot_wave_field_panels(BA)
if res is None:
    print("no wave output for this run (waves off, or no hm0)")

### Maximum flood depth — Shrewsbury / Navesink estuary

The estuary the leak used to drain. **After**, the back-bays fill.

In [ ]:
plots.plot_engine_panels(BA);

### Gauge & pre-storm tide

Read the **pre-storm tide** (left of the dotted "gauge dies" line): a tide floods *and* ebbs;
the broken run only ebbs, and at Shark it never oscillates at all (inlet was dammed).
Obs: Shark 1.52 m, rising 0.47 of the time → **after** 1.30 m, 0.54.

In [ ]:
plots.plot_gauge_verification(BA);

### USGS high-water-mark residuals

Model − obs: red = too high, blue = too low, ✕ = dry where a mark says wet. The estuary's
blue under-marks lift toward neutral after the fix.

In [ ]:
plots.plot_hwm_residual_panels(BA);

### FEMA MOTF flood extent

**Read the CSI, not the POD** — MOTF is a bathtub surface, and POD structurally rewards
over-flooding. before CSI 0.51 → **after 0.64**, with FAR *down* 0.17 → 0.14.

In [ ]:
plots.plot_motf_panels(BA);

## The decision — the sealed premier 2×2

`sealed_faber_waves` is the **adopted premier**: gauge within 0.10 m of the surveyed crest,
Shark tide alive, best MOTF. **Faber over Galibier** — identical without waves, but
Galibier+waves overshoots hard (gauge +0.57 m, HWM bias +0.97), so Galibier is unofficially
retired.

⚠️ **Locality caveat:** the open coast that never broke drifted ~0.1 m on the rebuild
(south_coast −0.055 → +0.048), so the fix is not *purely* local — small next to the gains,
but worth a look.

In [ ]:
# The sealed premier 2x2 (tide, gauge, per-basin HWM bias).
# Produced by scripts/analyze_sealed.py; re-run that script if the CSV is stale.
sealed_csv = ROOT / "reports" / "sealed_premier.csv"
if sealed_csv.exists():
    sealed = pd.read_csv(sealed_csv)
    cols = [c for c in ["desc", "shark_frac_rising", "shark_tide", "shrews_tide",
                        "gauge", "gauge_err", "shrewsbury_navesink", "shark_river",
                        "south_coast", "atlantic_oceanfront", "rmse"] if c in sealed]
    display(sealed[cols].round(3))
    print("\nADOPTED: sealed_faber_waves (Faber, waves on).")
    print("Locality caveat: south_coast drifted -0.055 -> +0.048 on the rebuild.")
else:
    print("run:  NJ_ROOT=$PWD PYTHONPATH=$PWD python scripts/analyze_sealed.py")